# Ablation: does pretraining the conditional pose VAE help? (A\*02:01, 6 peptides)

Fixed antigen / single allele. Warm-started TCR + MHC-conditioned peptide + MHC towers;
only the **conditional pose VAE** handling differs across conditions:

- **A** cpose trained by classification only (no aux) — floor
- **B** cpose trained *jointly* with aux VAE loss, no pretrain — baseline to beat
- **C** Scheme-1 pretrain cpose → finetune with aux — candidate
- **D** pretrain cpose → freeze cpose+tcr_proj, train head only — probe (is the pretrained
  pose latent discriminative on its own?)

Evaluated on **LOPO** (unseen peptide) and **TCR-cluster** (unseen receptor) splits.
Decision: pretraining helps iff **C > B** (paired bootstrap CI excludes 0), esp. on LOPO.
Towers are cached once per split (frozen), so this runs cheaply. Run on Colab (GPU).

In [ ]:
import os, sys, subprocess
REPO = '/content/tcrpmhc_pose_binding'
REPO_URL = 'github.com/92kunheekim/tcrpmhc_pose_binding.git'
from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', f'https://x-access-token:{TOKEN}@{REPO_URL}', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
os.chdir(REPO); sys.path.insert(0, f'{REPO}/src')
DATA_DIR = f'{REPO}/data'; PRETRAIN_DIR = f'{REPO}/pretrained'
RESULTS = f'{REPO}/results'; os.makedirs(RESULTS, exist_ok=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements.txt'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers'])
import json, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from data import load_data, repeated_splits, cluster_tcrs
from train_utils import make_warm_start, aggregate_oof, cluster_bootstrap, DEVICE
from pan_specific import compute_esm_embeddings, run_cpose_ablation
print('device:', DEVICE)

## Config

In [ ]:
ALLELE     = 'HLA-A*02:01'
HEAD_FEATS = 'pose_tp'      # pose + t*p (single allele -> drop h/th)
PEP_ARCH   = 'transformer'  # must match the pretrained peptide encoder
EPOCHS, PRE_EPOCHS = 15, 30
N_REPEATS  = 10            # TCR-cluster repeated 80/20 splits
ESM_MAXLEN = 190

## 1. Data + warm-start + single-allele ESM

In [ ]:
D = load_data(f'{DATA_DIR}/10x_cd8_A0201_6pep', min_iptm=0.5)
pool, trans_scale = D['pool'], D['trans_scale']
print('rows:', len(pool), '| peptides:', sorted(pool.peptide.unique()))

# single-allele groove ESM: map A*02:01 -> alpha1alpha2 -> one ESM-2 vector for all rows
a1a2 = json.load(open(f'{DATA_DIR}/pmhc_classI_pretraining/processed/mhc_classI_a1a2.json'))
groove = a1a2[ALLELE]
pool['mhc_seq'] = groove
esm_table = compute_esm_embeddings([groove], maxlen=ESM_MAXLEN)

# warm-start towers from the pretrained encoders
TCR_ENC   = torch.load(f'{PRETRAIN_DIR}/tcr_encoders_corpus.pt', map_location='cpu')
pep_state = torch.load(f'{PRETRAIN_DIR}/pep_encoder_masked.pt', map_location='cpu')
mhc_state = torch.load(f'{PRETRAIN_DIR}/mhc_encoder_masked.pt', map_location='cpu')
warm = make_warm_start(TCR_ENC, pep_state=pep_state, mhc_state=mhc_state)

## 2. Splits — LOPO (unseen peptide) and TCR-cluster (unseen receptor)

In [ ]:
peps = pool.peptide.to_numpy()
lopo = [(np.where(peps != q)[0], np.where(peps == q)[0]) for q in sorted(pool.peptide.unique())]
clu  = repeated_splits(pool, n_repeats=N_REPEATS)
print('LOPO splits:', len(lopo), '| cluster splits:', len(clu))

## 3. Run the ablation

In [ ]:
m_lopo, oof_lopo = run_cpose_ablation(pool, trans_scale, esm_table, lopo, warm,
                                     head_feats=HEAD_FEATS, pep_arch=PEP_ARCH,
                                     epochs=EPOCHS, pre_epochs=PRE_EPOCHS)
m_clu, oof_clu = run_cpose_ablation(pool, trans_scale, esm_table, clu, warm,
                                    head_feats=HEAD_FEATS, pep_arch=PEP_ARCH,
                                    epochs=EPOCHS, pre_epochs=PRE_EPOCHS)
m_lopo['eval'] = 'LOPO'; m_clu['eval'] = 'cluster'
metrics = pd.concat([m_lopo, m_clu], ignore_index=True)
metrics.to_csv(f'{RESULTS}/cpose_ablation_per_split.csv', index=False)

## 4. Summary (mean ± std across splits)

In [ ]:
summary = (metrics.groupby(['eval', 'condition'])[['auroc', 'auprc']]
           .agg(['mean', 'std']).round(3))
print(summary)
summary.to_csv(f'{RESULTS}/cpose_ablation_summary.csv')

## 5. Paired cluster-bootstrap: C − B (per-sample aggregated, TCR-cluster eval)

The decisive test. Positive CI excluding 0 ⇒ pretraining the conditional pose VAE adds
value beyond the joint aux-loss baseline.

In [ ]:
groups = cluster_tcrs(pool); y = pool.label.to_numpy().astype(int); n = len(pool)
def agg(oof, cond):
    pa, tested = aggregate_oof(oof[cond]['idx'], oof[cond]['p'], n); return pa, tested
for ctr in [('C', 'B'), ('B', 'A'), ('C', 'A')]:
    pa, ta = agg(oof_clu, ctr[0]); pb, tb = agg(oof_clu, ctr[1]); tested = ta & tb
    (dR, loR, hiR, pR), (dP, loP, hiP, pP) = cluster_bootstrap(y, pa, pb, groups, tested)
    print(f'{ctr[0]}-{ctr[1]}:  dAUROC={dR:+.3f} [{loR:+.3f},{hiR:+.3f}] p={pR:.3f}   '
          f'dAUPRC={dP:+.3f} [{loP:+.3f},{hiP:+.3f}] p={pP:.3f}')

## 6. Plot

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
order = ['A', 'B', 'C', 'D']
for j, ev in enumerate(['LOPO', 'cluster']):
    sub = metrics[metrics.eval == ev]
    mean = sub.groupby('condition').auroc.mean().reindex(order)
    sd   = sub.groupby('condition').auroc.std().reindex(order)
    base = sub.prevalence.mean()
    ax[j].bar(order, mean.values, yerr=sd.values, capsize=4)
    ax[j].axhline(0.5, ls='--', c='gray')
    ax[j].set(title=f'{ev}: AUROC by condition', ylim=(0.4, 1.0), ylabel='AUROC')
plt.tight_layout(); plt.savefig(f'{RESULTS}/cpose_ablation.png', dpi=150); plt.show()

## 7. Commit & push

In [ ]:
import json
for _nb in ['notebooks/ablation_cpose.ipynb']:
    _p = f'{REPO}/{_nb}'
    if os.path.exists(_p):
        _d = json.load(open(_p)); _d.get('metadata', {}).pop('widgets', None)
        for _c in _d.get('cells', []): _c.get('metadata', {}).pop('widgets', None)
        json.dump(_d, open(_p, 'w'), indent=1)
subprocess.run(['git', 'config', 'user.email', '92kunheekim@gmail.com'], cwd=REPO)
subprocess.run(['git', 'config', 'user.name', 'KH Kim'], cwd=REPO)
subprocess.run(['git', 'add', '-f', 'results', 'notebooks/ablation_cpose.ipynb'], cwd=REPO)
subprocess.run(['git', 'commit', '-m', 'Conditional pose-VAE pretraining ablation (A0201 6pep) [colab]'], cwd=REPO)
subprocess.run(['git', 'push', 'origin', 'main'], cwd=REPO, check=True)
print('pushed.')